In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, dayofweek, date_format
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Create Spark session
spark = SparkSession.builder.appName("Lab4_Spark_MLlib").getOrCreate()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
path = "/content/drive/MyDrive/Colab Notebooks/tp4_data/tp4_data/*.csv"
df = spark.read.csv(path, header=True, inferSchema=True)

Mounted at /content/drive


In [ ]:
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [ ]:
df = df.fillna(0)
df.show(5)

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   537226|    22811|SET OF 6 T-LIGHTS...|       6|2010-12-06 08:34:00|     2.95|   15987.0|United Kingdom|
|   537226|    21713|CITRONELLA CANDLE...|       8|2010-12-06 08:34:00|      2.1|   15987.0|United Kingdom|
|   537226|    22927|GREEN GIANT GARDE...|       2|2010-12-06 08:34:00|     5.95|   15987.0|United Kingdom|
|   537226|    20802|SMALL GLASS SUNDA...|       6|2010-12-06 08:34:00|     1.65|   15987.0|United Kingdom|
|   537226|    22052|VINTAGE CARAVAN G...|      25|2010-12-06 08:34:00|     0.42|   15987.0|United Kingdom|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
only showing top 5 rows


In [ ]:
from pyspark.sql.functions import date_format, to_date, col

# Ensure InvoiceDate is in date format
df = df.withColumn("InvoiceDate", to_date(col("InvoiceDate"), "yyyy-MM-dd"))

# Add day_of_week column
df = df.withColumn("day_of_week", date_format(col("InvoiceDate"), "EEEE"))

df.select("InvoiceDate", "day_of_week").distinct().show(5)

+-----------+-----------+
|InvoiceDate|day_of_week|
+-----------+-----------+
| 2010-12-09|   Thursday|
| 2010-12-08|  Wednesday|
| 2010-12-07|    Tuesday|
| 2010-12-13|     Monday|
| 2010-12-01|  Wednesday|
+-----------+-----------+
only showing top 5 rows


In [ ]:
train_df = df.filter(col("InvoiceDate") < "2010-12-13")
test_df = df.filter(col("InvoiceDate") >= "2010-12-13")

print(f"Training set count: {train_df.count()}")
print(f"Test set count: {test_df.count()}")

Training set count: 26732
Test set count: 18676


In [ ]:
from pyspark.ml.feature import StringIndexer

# Create the StringIndexer
day_indexer = StringIndexer(inputCol="day_of_week", outputCol="day_of_week_encoded")

# Fit the indexer on the training data
day_indexer_model = day_indexer.fit(train_df)

# Transform the training data
train_df_indexed = day_indexer_model.transform(train_df)

# Show the result
train_df_indexed.select("day_of_week", "day_of_week_encoded").distinct().show()

+-----------+-------------------+
|day_of_week|day_of_week_encoded|
+-----------+-------------------+
|   Thursday|                1.0|
|  Wednesday|                0.0|
|     Sunday|                3.0|
|    Tuesday|                5.0|
|     Friday|                2.0|
|     Monday|                4.0|
+-----------+-------------------+



StringIndexer maps each unique string to a numeric index.
The most frequent day in the training set becomes 0.0, the next 1.0, et

The solution: One-Hot Encoding

Convert the numeric index into a vector of 0s and 1s, where each category is independent.

Example:

| day_of_week | one_hot_vector  |
| ----------- | --------------- |
| Monday      | [1,0,0,0,0,0,0] |
| Tuesday     | [0,1,0,0,0,0,0] |
| Wednesday   | [0,0,1,0,0,0,0] |
| Saturday    | [0,0,0,0,0,1,0] |


In [ ]:
from pyspark.ml.feature import OneHotEncoder

# Convert day_of_week_encoded → day_of_week_vector
day_encoder = OneHotEncoder(inputCol="day_of_week_encoded", outputCol="day_of_week_vector")

# Fit on training data and transform
train_df_encoded = day_encoder.fit(train_df_indexed).transform(train_df_indexed)

# Show distinct rows with day name, encoded number, and vector
train_df_encoded.select("day_of_week", "day_of_week_encoded", "day_of_week_vector").distinct().show(6, truncate=False)

+-----------+-------------------+------------------+
|day_of_week|day_of_week_encoded|day_of_week_vector|
+-----------+-------------------+------------------+
|Thursday   |1.0                |(5,[1],[1.0])     |
|Tuesday    |5.0                |(5,[],[])         |
|Friday     |2.0                |(5,[2],[1.0])     |
|Monday     |4.0                |(5,[4],[1.0])     |
|Sunday     |3.0                |(5,[3],[1.0])     |
|Wednesday  |0.0                |(5,[0],[1.0])     |
+-----------+-------------------+------------------+



(5,[1],[1.0]) → [0.0, 1.0, 0.0, 0.0, 0.0]

In [ ]:
from pyspark.ml.feature import VectorAssembler

# Combine UnitPrice, Quantity, and day_of_week_encoded into a single features vector
assembler = VectorAssembler(
    inputCols=["UnitPrice", "Quantity", "day_of_week_encoded"],  # numeric columns
    outputCol="features"
)

# Transform the training data
train_df_features = assembler.transform(train_df_indexed)

# Show sample output
train_df_features.select("UnitPrice", "Quantity", "day_of_week_encoded", "features").show(5, truncate=False)

+---------+--------+-------------------+---------------+
|UnitPrice|Quantity|day_of_week_encoded|features       |
+---------+--------+-------------------+---------------+
|2.95     |6       |4.0                |[2.95,6.0,4.0] |
|2.1      |8       |4.0                |[2.1,8.0,4.0]  |
|5.95     |2       |4.0                |[5.95,2.0,4.0] |
|1.65     |6       |4.0                |[1.65,6.0,4.0] |
|0.42     |25      |4.0                |[0.42,25.0,4.0]|
+---------+--------+-------------------+---------------+
only showing top 5 rows


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# Step 6: StringIndexer (convert day_of_week → numeric index)
day_indexer = StringIndexer(inputCol="day_of_week", outputCol="day_of_week_encoded")

# Step 7: OneHotEncoder (optional, convert numeric index → one-hot vector)
day_encoder = OneHotEncoder(inputCol="day_of_week_encoded", outputCol="day_of_week_vector")

# Step 8: VectorAssembler (combine features)
assembler = VectorAssembler(
    inputCols=["UnitPrice", "Quantity", "day_of_week_encoded"],  # or "day_of_week_vector" if using one-hot
    outputCol="features"
)

# Create the pipeline
pipeline = Pipeline(stages=[day_indexer, day_encoder, assembler])

What Spark does

By default, StringIndexer will throw an error if it encounters a new/unseen value.

To handle this, you can set:

day_indexer = StringIndexer(
    inputCol="day_of_week",
    outputCol="day_of_week_encoded",
    handleInvalid="keep"   # <-- important
)

handleInvalid="keep" → assigns all unseen values to an extra index (like 6.0 in this example).

Other options:

"skip" → ignores rows with unseen values.

"error" → default behavior, raises an error.

In [ ]:
# Fit the pipeline on the training data
pipeline_model = pipeline.fit(train_df)  # learn StringIndexer, OneHotEncoder, VectorAssembler

# Transform the training data
train_prepared = pipeline_model.transform(train_df)

# Show the results
train_prepared.select(
    "UnitPrice", "Quantity", "day_of_week", "day_of_week_encoded", "day_of_week_vector", "features"
).show(5, truncate=False)

+---------+--------+-----------+-------------------+------------------+---------------+
|UnitPrice|Quantity|day_of_week|day_of_week_encoded|day_of_week_vector|features       |
+---------+--------+-----------+-------------------+------------------+---------------+
|2.95     |6       |Monday     |4.0                |(5,[4],[1.0])     |[2.95,6.0,4.0] |
|2.1      |8       |Monday     |4.0                |(5,[4],[1.0])     |[2.1,8.0,4.0]  |
|5.95     |2       |Monday     |4.0                |(5,[4],[1.0])     |[5.95,2.0,4.0] |
|1.65     |6       |Monday     |4.0                |(5,[4],[1.0])     |[1.65,6.0,4.0] |
|0.42     |25      |Monday     |4.0                |(5,[4],[1.0])     |[0.42,25.0,4.0]|
+---------+--------+-----------+-------------------+------------------+---------------+
only showing top 5 rows


In [ ]:
from pyspark.ml.clustering import KMeans

# Create KMeans instance
kmeans = KMeans(
    k=20,             # Number of clusters
    seed=42,          # Optional: for reproducibility
    featuresCol="features",  # Column with feature vectors
    predictionCol="cluster"  # Output column for cluster assignment
)

In [ ]:
# Train KMeans on the prepared training data
kmeans_model = kmeans.fit(train_prepared)

# Print the cluster centers
print("Cluster Centers:")
for center in kmeans_model.clusterCenters():
    print(center)

Cluster Centers:
[ 1.86144201 15.30116435  1.86206897]
[ 1.3524695e+04 -5.0000000e-01  5.0000000e+00]
[ 3.00e-02 -9.36e+03  1.00e+00]
[ 6.70671e+03 -1.00000e+00  5.00000e+00]
[886.762   1.      3.   ]
[1.80e-01 2.88e+03 1.00e+00]
[  2.00702247 100.83146067   1.78089888]
[3.600e-01 1.776e+03 2.000e+00]
[1.0800e+00 1.4305e+03 1.2500e+00]
[212.10083333   0.5          2.        ]
[ 5.51925e+03 -1.00000e+00  5.00000e+00]
[4.17277217 2.3196582  2.23581343]
[  1.912 992.      4.   ]
[  1.54743243 172.72972973   1.58108108]
[595.95636364   0.81818182   2.63636364]
[   0.62428571 -126.53571429    1.71428571]
[  2.01884615 356.5          2.26923077]
[   0. -723.    1.]
[ 1.98291667 46.40555556  1.94444444]
[  2.37 624.     2.8 ]


In [ ]:
# Transform the test data using the pipeline first
test_prepared = pipeline_model.transform(test_df)

# Predict clusters using the trained KMeans model
test_predictions = kmeans_model.transform(test_prepared)

# Show sample predictions
test_predictions.select("UnitPrice", "Quantity", "day_of_week", "features", "cluster").show(5, truncate=False)

+---------+--------+-----------+---------------+-------+
|UnitPrice|Quantity|day_of_week|features       |cluster|
+---------+--------+-----------+---------------+-------+
|4.95     |3       |Friday     |[4.95,3.0,2.0] |11     |
|3.95     |4       |Friday     |[3.95,4.0,2.0] |11     |
|0.42     |12      |Friday     |[0.42,12.0,2.0]|0      |
|0.85     |12      |Friday     |[0.85,12.0,2.0]|0      |
|0.85     |12      |Friday     |[0.85,12.0,2.0]|0      |
+---------+--------+-----------+---------------+-------+
only showing top 5 rows


In [ ]:
from pyspark.ml.evaluation import ClusteringEvaluator

# Create a ClusteringEvaluator
evaluator = ClusteringEvaluator(
    featuresCol="features",    # column with features used for clustering
    predictionCol="cluster",   # column with cluster IDs
    metricName="silhouette",   # metric to calculate
    distanceMeasure="squaredEuclidean"  # distance used in KMeans
)

# Calculate silhouette score on the test set
silhouette = evaluator.evaluate(test_predictions)

print(f"Silhouette coefficient for test set: {silhouette}")

Silhouette coefficient for test set: 0.7271557607067531


The Silhouette score measures how well each point fits into its cluster:

+1 → point is well matched to its cluster and far from others

0 → point is on or near the decision boundary between clusters

-1 → point might be assigned to the wrong cluster